# Lab 02 — Lazy Evaluation and Actions

**Tujuan:** Membedakan transformasi yang membangun plan dengan action yang mengeksekusi.

**Lingkungan:** Spark standalone (`spark-master:7077`), Jupyter, dua worker. Jalankan sel secara berurutan. Setiap notebook dapat dijalankan sendiri.

**Metode:** Prediksi → jalankan → amati `explain()`/Spark UI → ubah satu variabel → simpulkan.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Spark Training - Lab 02")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "4")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/10 00:36:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.7
Master: spark://spark-master:7077


## 1. Build a plan without action

**Prediksi:** apakah data sudah dihitung setelah ketiga baris ini?

In [6]:
df = spark.range(0, 2_000_000, 1, 4)
filtered = df.filter("id > 500000")
even = filtered.filter("id % 2 = 0")
result = even.select("id")
print("Transformations defined")

Transformations defined


Buka Spark UI; lihat apakah **job baru** muncul dari sel sebelumnya.

In [7]:
result.explain("formatted")

== Physical Plan ==
* Filter (2)
+- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#24L]
Arguments: Range (0, 2000000, step=1, splits=Some(4))

(2) Filter [codegen id : 1]
Input [1]: [id#24L]
Condition : ((id#24L > 500000) AND ((id#24L % 2) = 0))




## 2. Trigger action

In [9]:
result.count()

749999

## 3. Compare actions

`show()` dan `count()` adalah action, tetapi bentuk plan dan jumlah pekerjaan dapat berbeda.

In [10]:
result.show(5)
print("Count:", result.count())

+------+
|    id|
+------+
|500002|
|500004|
|500006|
|500008|
|500010|
+------+
only showing top 5 rows

Count: 749999


**Tugas:** ganti kondisi menjadi `id % 10 = 0`. Perkirakan hasil `count()` sebelum menjalankan.

## Refleksi peserta

Tuliskan: **(1)** apa prediksi Anda, **(2)** apa yang teramati di Spark UI atau output, **(3)** mengapa Spark berperilaku demikian. Hindari menyimpulkan dari waktu eksekusi satu kali saja.

In [11]:
spark.stop()